# Motor de proyección — IPC y remesas familiares

Cuaderno de ejecución del componente predictivo de la tesis.
Corresponde a las etapas 2 a 5 del apartado 4.6.

Antes de correrlo, suba la carpeta `sistema/` a su Google Drive o clone el repositorio.

In [ ]:
!pip install -q xgboost shap statsmodels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajuste esta ruta a donde haya dejado la carpeta del proyecto
RUTA = '/content/drive/MyDrive/sistema'

import sys
sys.path.insert(0, RUTA)

## 1. Carga y depuración de las series

Ajuste `filas_encabezado` hasta que el diagnóstico muestre el número de
observaciones esperado y ningún mes ausente.

In [ ]:
from src import carga

ipc = carga.desde_matriz_anio_mes(
    f'{RUTA}/datos/crudos/ipc_base1999.xlsx',
    nombre='IPC',
    filas_encabezado=0,
)
carga.diagnostico(ipc)

In [ ]:
remesas = carga.desde_matriz_anio_mes(
    f'{RUTA}/datos/crudos/remesas.xlsx',
    nombre='Remesas familiares',
    filas_encabezado=0,
)
carga.diagnostico(remesas)
# Si el cuadro trae una fila por mes y una columna por año, use
# carga.desde_matriz_mes_anio con los mismos parámetros.

In [ ]:
carga.guardar(ipc, f'{RUTA}/datos/procesados/ipc.csv')
carga.guardar(remesas, f'{RUTA}/datos/procesados/remesas.csv')

## 2. Caracterización de las series (apartado 5.1)

In [ ]:
import matplotlib.pyplot as plt

fig, ejes = plt.subplots(2, 1, figsize=(11, 7))
ipc.plot(ax=ejes[0], color='#1f4e79')
ejes[0].set_title('Índice de precios al consumidor'); ejes[0].set_xlabel('')
remesas.plot(ax=ejes[1], color='#7d3c98')
ejes[1].set_title('Remesas familiares (millones de US$)'); ejes[1].set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

for serie in (ipc, remesas):
    est, p, *_ = adfuller(serie.dropna(), autolag='AIC')
    print(f'{serie.name:22s} ADF={est:8.4f}  p={p:.4f}  '
          f'{"estacionaria" if p < 0.05 else "no estacionaria"}')

## 3. Experimento sobre el IPC

Tarda varios minutos: el ARIMA se reestima en cada origen de la validación.

In [ ]:
from src import experimento

salida_ipc = experimento.ejecutar(ipc, horizontes=(1, 3, 6, 12), logaritmo=False)

In [ ]:
salida_ipc['desempeno']

In [ ]:
salida_ipc['diebold_mariano']

In [ ]:
print(salida_ipc['sobreajuste'])
print('Especificación ARIMA:', salida_ipc['especificacion_arima'])

## 4. Experimento sobre las remesas

Aquí se activa la transformación logarítmica por el crecimiento multiplicativo
de la serie. Se aplica al ARIMA y al XGBoost por igual, y las proyecciones se
devuelven a millones de dólares antes de calcular los errores.

In [ ]:
salida_rem = experimento.ejecutar(remesas, horizontes=(1, 3, 6, 12), logaritmo=True)

In [ ]:
salida_rem['desempeno']

In [ ]:
salida_rem['diebold_mariano']

## 5. Explicabilidad con SHAP (apartado 3.4.2)

Se entrena el XGBoost con la serie completa a un mes y se descompone su
proyección. La importancia global sirve para el Capítulo 5; la cascada explica
la proyección del próximo mes.

In [ ]:
from src import modelos, explicabilidad

for serie, log in ((ipc, False), (remesas, True)):
    modelo = modelos.Xgboost(logaritmo=log).ajustar(serie, 1)
    display(explicabilidad.importancia_global(modelo))
    local = explicabilidad.explicacion_local(modelo)
    print(explicabilidad.texto_explicativo(local))
    explicabilidad.grafico_importancia(explicabilidad.importancia_global(modelo), serie.name)
    explicabilidad.grafico_cascada(local, f'Proyección de {local["fecha_objetivo"]:%m/%Y}')
    plt.show()

## 6. Exportación de resultados

La forma más simple de dejar todo listo para la aplicación es correr
`python ejecutar_todo.py` en VS Code. Esta celda guarda solo las tablas.

In [ ]:
experimento.guardar_resultados(salida_ipc, f'{RUTA}/resultados', 'ipc')
experimento.guardar_resultados(salida_rem, f'{RUTA}/resultados', 'remesas')
print('Resultados guardados.')